In [0]:
# ============================================================
# JOB MARKET ANALYSIS PIPELINE
# Greenhouse API → AWS S3 RAW
# ============================================================

import requests
import json
from datetime import datetime, timezone


# ============================================================
# 1. CONFIGURATION
# ============================================================

# AWS S3 RAW location
S3_RAW_PATH = "s3://job-market-analysis-pipeline/raw-data/greenhouse"


# Companies that passed API verification
GREENHOUSE_COMPANIES = {
    "Stripe": "stripe",
    "Cloudflare": "cloudflare",
    "Figma": "figma",
    "Coinbase": "coinbase",
    "GuidePoint Security": "guidepointsecurity"
}


# ============================================================
# 2. CURRENT INGESTION DATE
# ============================================================

ingestion_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")

run_timestamp = datetime.now(timezone.utc).isoformat()

print(f"Ingestion date: {ingestion_date}")
print(f"Run timestamp: {run_timestamp}")


# ============================================================
# 3. GREENHOUSE API FUNCTION
# ============================================================

def fetch_greenhouse_jobs(company_name, board_token):

    url = (
        f"https://boards-api.greenhouse.io/"
        f"v1/boards/{board_token}/jobs"
    )

    params = {
        "content": "true"
    }

    print("\n------------------------------------------")
    print(f"Fetching: {company_name}")
    print(f"Endpoint: {url}")

    try:

        response = requests.get(
            url,
            params=params,
            timeout=30
        )

        print(f"HTTP Status: {response.status_code}")

        response.raise_for_status()

        data = response.json()

        # Validate response structure
        if "jobs" not in data:

            raise ValueError(
                f"'jobs' field not found for {company_name}"
            )

        jobs = data["jobs"]

        if not isinstance(jobs, list):

            raise ValueError(
                f"'jobs' is not a list for {company_name}"
            )

        print(f"Jobs received: {len(jobs)}")

        return data

    except requests.exceptions.Timeout:

        print(
            f"ERROR: Request timeout - "
            f"{company_name}"
        )

        return None

    except requests.exceptions.RequestException as e:

        print(
            f"ERROR: API request failed - "
            f"{company_name}"
        )

        print(str(e))

        return None

    except json.JSONDecodeError:

        print(
            f"ERROR: Invalid JSON response - "
            f"{company_name}"
        )

        return None

    except Exception as e:

        print(
            f"ERROR: {company_name}"
        )

        print(str(e))

        return None


# ============================================================
# 4. INGESTION RESULT TRACKING
# ============================================================

successful_companies = []
failed_companies = []


# ============================================================
# 5. FETCH API DATA AND WRITE TO S3
# ============================================================

for company_name, board_token in GREENHOUSE_COMPANIES.items():

    # --------------------------------------------------------
    # Fetch API data
    # --------------------------------------------------------

    data = fetch_greenhouse_jobs(
        company_name,
        board_token
    )

    # --------------------------------------------------------
    # Handle failed API request
    # --------------------------------------------------------

    if data is None:

        failed_companies.append(company_name)

        continue


    # --------------------------------------------------------
    # Build RAW payload
    #
    # We preserve the original API job records and add
    # ingestion metadata at the outer level.
    # --------------------------------------------------------

    output = {

        "source": "greenhouse",

        "company": company_name,

        "board_token": board_token,

        "ingestion_timestamp": run_timestamp,

        "ingestion_date": ingestion_date,

        "job_count": len(
            data.get("jobs", [])
        ),

        "jobs": data.get(
            "jobs",
            []
        )
    }


    # --------------------------------------------------------
    # Convert payload to JSON
    # --------------------------------------------------------

    json_data = json.dumps(
        output,
        ensure_ascii=False
    )


    # --------------------------------------------------------
    # Build S3 path
    # --------------------------------------------------------

    output_path = (
        f"{S3_RAW_PATH}/"
        f"ingestion_date={ingestion_date}/"
        f"{board_token}.json"
    )


    # --------------------------------------------------------
    # Write JSON to AWS S3
    # --------------------------------------------------------

    try:

        dbutils.fs.put(
            output_path,
            json_data,
            overwrite=True
        )

        print(
            f"RAW data written successfully:"
            f"\n{output_path}"
        )


        successful_companies.append({

            "company": company_name,

            "jobs": len(
                data.get("jobs", [])
            ),

            "path": output_path

        })


    except Exception as e:

        print(
            f"ERROR writing "
            f"{company_name} to S3"
        )

        print(str(e))

        failed_companies.append(
            company_name
        )


# ============================================================
# 6. INGESTION SUMMARY
# ============================================================

print("\n")
print("==========================================")
print("GREENHOUSE INGESTION SUMMARY")
print("==========================================")

print(
    f"Successful companies: "
    f"{len(successful_companies)}"
)

print(
    f"Failed companies: "
    f"{len(failed_companies)}"
)


# ============================================================
# 7. SUCCESSFUL COMPANIES
# ============================================================

print("\nSuccessful:")

for item in successful_companies:

    print(
        f"{item['company']} → "
        f"{item['jobs']} jobs → "
        f"{item['path']}"
    )


# ============================================================
# 8. FAILED COMPANIES
# ============================================================

print("\nFailed:")

if failed_companies:

    for company in failed_companies:

        print(company)

else:

    print("None")


# ============================================================
# 9. FINAL STATUS
# ============================================================

print("\n==========================================")

if len(failed_companies) == 0:

    print(
        "GREENHOUSE INGESTION COMPLETED "
        "SUCCESSFULLY"
    )

else:

    print(
        "GREENHOUSE INGESTION COMPLETED "
        "WITH FAILURES"
    )

print("==========================================")

Ingestion date: 2026-10-06
Run timestamp: 2026-10-06T12:44:37.740098+00:00

------------------------------------------
Fetching: Stripe
Endpoint: https://boards-api.greenhouse.io/v1/boards/stripe/jobs
HTTP Status: 200
Jobs received: 719
Wrote 4998477 bytes.
RAW data written successfully:
s3://job-market-analysis-pipeline/raw-data/greenhouse/ingestion_date=2026-10-06/stripe.json

------------------------------------------
Fetching: Cloudflare
Endpoint: https://boards-api.greenhouse.io/v1/boards/cloudflare/jobs
HTTP Status: 200
Jobs received: 416
Wrote 6301383 bytes.
RAW data written successfully:
s3://job-market-analysis-pipeline/raw-data/greenhouse/ingestion_date=2026-10-06/cloudflare.json

------------------------------------------
Fetching: Figma
Endpoint: https://boards-api.greenhouse.io/v1/boards/figma/jobs
HTTP Status: 200
Jobs received: 161
Wrote 1594044 bytes.
RAW data written successfully:
s3://job-market-analysis-pipeline/raw-data/greenhouse/ingestion_date=2026-10-06/figma.jso